# Transformer on TinyStories

An encoder-decoder Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)), built on the structure of
[The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/), trained to continue short stories from the
[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) dataset.

The model sees three 32-token chunks of a story (96 tokens) through the encoder and predicts the next 32-token chunk with the decoder.

- Byte-level BPE tokenizer (8,000 vocab) trained on TinyStories
- Fixed-shape windows (96-token source, 32-token target), so batches need no padding
- 4 encoder + 4 decoder layers, `d_model=384`, 6 heads, `d_ff=1408` (~25M parameters)
- AdamW with the Noam warmup / inverse-sqrt decay schedule from the paper
- Mixed-precision training, checkpointing, train/val loss and perplexity
- Greedy and top-k sampled decoding

## Setup and configuration

In [ ]:
import os

# Disable HF progress-bar widgets (avoids ipywidgets errors in headless Jupyter)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    from datasets import load_dataset
    import datasets as hf_datasets
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders
    from tqdm import tqdm
except ImportError:
    import sys
    import subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "datasets", "tokenizers", "tqdm"]
    )
    from datasets import load_dataset
    import datasets as hf_datasets
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders
    from tqdm import tqdm

hf_datasets.disable_progress_bars()

import copy
import json
import math
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterator, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, IterableDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True  # input shapes are fixed, so let cuDNN autotune
torch.backends.cudnn.deterministic = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"  # mixed precision on GPU only

DATASET_NAME = "roneneldan/TinyStories"

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

CHUNK_SIZE = 32
SRC_CHUNKS = 3
SRC_LEN = CHUNK_SIZE * SRC_CHUNKS
TGT_LEN = CHUNK_SIZE

TOKENIZER_DIR = Path("./artifacts")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = TOKENIZER_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_PATH = TOKENIZER_DIR / "tinystories_bpe_tokenizer.json"

MAX_TRAIN_STORIES_FOR_TOKENIZER = None  # set to an int to speed up tokenizer training
MAX_TRAIN_STORIES = None                # set to an int to cap training windows
MAX_VAL_STORIES = None

# Dataloader settings
NUM_WORKERS = min(8, (os.cpu_count() or 4))
BATCH_SIZE = 128
PREFETCH_FACTOR = 4

@dataclass
class ModelConfig:
    src_vocab: int = 8000
    tgt_vocab: int = 8000
    N: int = 4
    d_model: int = 384
    d_ff: int = 1408
    h: int = 6
    dropout: float = 0.1

CONFIG = ModelConfig()


## Model

Pre-norm encoder-decoder with scaled dot-product multi-head attention, position-wise feed-forward layers and sinusoidal positional encodings.

In [ ]:
def clones(module: nn.Module, N: int) -> nn.ModuleList:
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])


class LayerNorm(nn.Module):
    def __init__(self, features: int, eps: float = 1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True, unbiased=False)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2


class SublayerConnection(nn.Module):
    def __init__(self, size: int, dropout: float):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))


class EncoderLayer(nn.Module):
    def __init__(self, size: int, self_attn, feed_forward, dropout: float):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = self.sublayer[0](x, lambda x_: self.self_attn(x_, x_, x_, mask))
        return self.sublayer[1](x, self.feed_forward)


class Encoder(nn.Module):
    def __init__(self, layer: nn.Module, N: int):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class DecoderLayer(nn.Module):
    def __init__(self, size: int, self_attn, src_attn, feed_forward, dropout: float):
        super().__init__()
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
        self.size = size

    def forward(self, x: torch.Tensor, memory: torch.Tensor, src_mask: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        x = self.sublayer[0](x, lambda x_: self.self_attn(x_, x_, x_, tgt_mask))
        x = self.sublayer[1](x, lambda x_: self.src_attn(x_, memory, memory, src_mask))
        return self.sublayer[2](x, self.feed_forward)


class Decoder(nn.Module):
    def __init__(self, layer: nn.Module, N: int):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x: torch.Tensor, memory: torch.Tensor, src_mask: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


def subsequent_mask(size: int, device: Optional[torch.device] = None) -> torch.Tensor:
    mask = torch.triu(torch.ones(1, size, size, dtype=torch.bool, device=device), diagonal=1)
    return ~mask


def attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
              mask: Optional[torch.Tensor] = None, dropout: Optional[nn.Module] = None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn


class MultiHeadAttention(nn.Module):
    def __init__(self, h: int, d_model: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]

        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(self.dropout(self.w1(x).relu()))


class Embeddings(nn.Module):
    def __init__(self, d_model: int, vocab: int):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.lut(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float, max_len: int = 4096):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)].detach()
        return self.dropout(x)


class Generator(nn.Module):
    def __init__(self, d_model: int, vocab: int):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src: torch.Tensor, tgt: torch.Tensor, src_mask: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory: torch.Tensor, src_mask: torch.Tensor, tgt: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)


def make_model(src_vocab: int, tgt_vocab: int, N: int = 4, d_model: int = 384, d_ff: int = 1408, h: int = 6, dropout: float = 0.1):
    c = copy.deepcopy
    attn = MultiHeadAttention(h, d_model, dropout)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)

    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )

    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model


## Data

Stream TinyStories, train (or load) a BPE tokenizer, and cut each story into sliding windows of 3 source chunks and 1 target chunk.

In [ ]:
def dataset_stream(split: str):
    """
    Try the official split first. If TinyStories does not expose that split in the
    current dataset revision, fall back to a deterministic split carved from train.
    """
    try:
        return load_dataset(DATASET_NAME, split=split, streaming=True)
    except Exception:
        base = load_dataset(DATASET_NAME, split="train", streaming=True)
        if split in {"validation", "valid", "val"}:
            return (row for i, row in enumerate(base) if i % 20 == 0)
        if split == "train":
            return (row for i, row in enumerate(base) if i % 20 != 0)
        raise


def iter_tiny_stories_texts(split: str, max_stories: Optional[int] = None) -> Iterator[str]:
    dataset = dataset_stream(split)
    for i, row in enumerate(dataset):
        text = row["text"]
        if text and text.strip():
            yield text
        if max_stories is not None and i + 1 >= max_stories:
            break


def train_or_load_tokenizer(tokenizer_path: Path = TOKENIZER_PATH) -> Tokenizer:
    if tokenizer_path.exists():
        return Tokenizer.from_file(str(tokenizer_path))

    tokenizer = Tokenizer(models.BPE(unk_token=UNK_TOKEN))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size=8000,
        min_frequency=2,
        special_tokens=[PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN],
        show_progress=True,
    )

    train_iter = iter_tiny_stories_texts("train", max_stories=MAX_TRAIN_STORIES_FOR_TOKENIZER)
    tokenizer.train_from_iterator(train_iter, trainer=trainer)
    tokenizer.save(str(tokenizer_path))
    return tokenizer


def encode_story(tokenizer: Tokenizer, text: str) -> List[int]:
    ids = tokenizer.encode(text).ids
    ids.append(EOS_ID)
    return ids


def story_to_windows(token_ids: Sequence[int], chunk_size: int = CHUNK_SIZE):
    n_chunks = len(token_ids) // chunk_size
    if n_chunks < 4:
        return
    for start_chunk in range(n_chunks - 3):
        src = token_ids[start_chunk * chunk_size : (start_chunk + 3) * chunk_size]
        tgt = token_ids[(start_chunk + 3) * chunk_size : (start_chunk + 4) * chunk_size]
        if len(src) == SRC_LEN and len(tgt) == TGT_LEN:
            yield src, tgt


class TinyStoriesWindowDataset(IterableDataset):
    def __init__(self, split: str, tokenizer: Tokenizer, max_stories: Optional[int] = None, seed: int = SEED, shuffle_buffer: int = 50_000):
        super().__init__()
        self.split = split
        self.tokenizer = tokenizer
        self.max_stories = max_stories
        self.seed = seed
        self.shuffle_buffer = shuffle_buffer

    def __iter__(self):
        dataset = dataset_stream(self.split)
        if self.split == "train" and hasattr(dataset, "shuffle"):
            dataset = dataset.shuffle(seed=self.seed, buffer_size=self.shuffle_buffer)

        story_count = 0
        for row in dataset:
            text = row["text"]
            if not text or not text.strip():
                continue
            token_ids = encode_story(self.tokenizer, text)
            for src, tgt in story_to_windows(token_ids):
                yield torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

            story_count += 1
            if self.max_stories is not None and story_count >= self.max_stories:
                break


def collate_windows(batch):
    src, tgt = zip(*batch)
    return torch.stack(src, dim=0), torch.stack(tgt, dim=0)


## Training utilities

In [ ]:
class Batch:
    def __init__(self, src: torch.Tensor, tgt: torch.Tensor, bos_id: int = BOS_ID, pad_id: int = PAD_ID):
        self.src = src
        self.tgt = tgt
        self.tgt_y = tgt
        self.src_mask = torch.ones(src.size(0), 1, src.size(1), dtype=torch.bool, device=src.device)
        bos = torch.full((tgt.size(0), 1), bos_id, dtype=torch.long, device=tgt.device)
        self.tgt_in = torch.cat([bos, tgt[:, :-1]], dim=1)
        self.tgt_mask = self.make_std_mask(self.tgt_in, pad_id)
        self.ntokens = self.tgt_y.numel()

    @staticmethod
    def make_std_mask(tgt: torch.Tensor, pad_id: int):
        tgt_mask = (tgt != pad_id).unsqueeze(1)
        tgt_mask = tgt_mask & subsequent_mask(tgt.size(1), device=tgt.device)
        return tgt_mask


@dataclass
class TrainState:
    step: int = 0
    accum_step: int = 0
    samples: int = 0
    tokens: int = 0


def build_dataloaders(
    tokenizer: Tokenizer,
    train_max_stories: Optional[int] = MAX_TRAIN_STORIES,
    val_max_stories: Optional[int] = MAX_VAL_STORIES,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
):
    train_ds = TinyStoriesWindowDataset("train", tokenizer, max_stories=train_max_stories)
    val_ds = TinyStoriesWindowDataset("validation", tokenizer, max_stories=val_max_stories)

    loader_kwargs = dict(
        batch_size=batch_size,
        collate_fn=collate_windows,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    train_loader = DataLoader(train_ds, **loader_kwargs)
    val_loader = DataLoader(val_ds, **loader_kwargs)
    return train_loader, val_loader


def build_optimizer_and_scheduler(model: nn.Module, d_model: int, warmup_steps: int = 4000, lr: float = 3e-4, weight_decay: float = 0.01):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9, weight_decay=weight_decay)

    def lr_lambda(step: int):
        step = max(1, step)
        scale = (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
        return scale / lr

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler


def compute_loss(logits: torch.Tensor, targets: torch.Tensor, criterion: nn.Module):
    vocab = logits.size(-1)
    return criterion(logits.reshape(-1, vocab), targets.reshape(-1))


def run_epoch(
    data_iter,
    model: EncoderDecoder,
    criterion,
    optimizer=None,
    scheduler=None,
    mode: str = "train",
    accum_iter: int = 1,
    grad_clip: float = 1.0,
    train_state: Optional[TrainState] = None,
    device: torch.device = DEVICE,
    scaler: Optional["torch.cuda.amp.GradScaler"] = None,
    amp_enabled: bool = AMP_ENABLED,
):
    if train_state is None:
        train_state = TrainState()

    is_train = mode in {"train", "train+log"}
    model.train(is_train)

    total_loss = 0.0
    total_tokens = 0
    start = time.time()

    if is_train and optimizer is not None:
        optimizer.zero_grad(set_to_none=True)

    for step, (src, tgt) in enumerate(data_iter, start=1):
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        batch = Batch(src, tgt)

        with torch.set_grad_enabled(is_train):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                out = model(batch.src, batch.tgt_in, batch.src_mask, batch.tgt_mask)
                logits = model.generator(out)
                loss_sum = compute_loss(logits, batch.tgt_y, criterion)
                loss = loss_sum / batch.ntokens

            if is_train:
                if scaler is not None and amp_enabled:
                    scaler.scale(loss / accum_iter).backward()
                else:
                    (loss / accum_iter).backward()

                if step % accum_iter == 0:
                    if scaler is not None and amp_enabled:
                        if grad_clip is not None:
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        if grad_clip is not None:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                        optimizer.step()

                    if scheduler is not None:
                        scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                    train_state.accum_step += 1

                train_state.step += 1
                train_state.samples += src.size(0)
                train_state.tokens += batch.ntokens

        total_loss += loss_sum.item()
        total_tokens += batch.ntokens

        if is_train and step % 40 == 0:
            elapsed = max(1e-9, time.time() - start)
            avg_loss = total_loss / total_tokens
            ppl = math.exp(min(20, avg_loss))
            lr = optimizer.param_groups[0]["lr"] if optimizer is not None else 0.0
            print(f"step {step:6d} | loss/token {avg_loss:7.4f} | ppl {ppl:8.2f} | tok/s {total_tokens / elapsed:8.1f} | lr {lr:.3e}")

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20, avg_loss))
    return {"loss": avg_loss, "ppl": ppl, "tokens": total_tokens, "train_state": train_state}


def save_checkpoint(path: Path, model: nn.Module, tokenizer_path: Path, optimizer=None, scheduler=None, config: Optional[dict] = None, history: Optional[list] = None):
    payload = {
        "model_state_dict": model.state_dict(),
        "tokenizer_path": str(tokenizer_path),
        "config": config or {},
        "history": history or [],
    }
    if optimizer is not None:
        payload["optimizer_state_dict"] = optimizer.state_dict()
    if scheduler is not None:
        payload["scheduler_state_dict"] = scheduler.state_dict()
    torch.save(payload, path)


def load_checkpoint(path: Path, model: nn.Module, optimizer=None, scheduler=None, map_location: str = "cpu"):
    payload = torch.load(path, map_location=map_location)
    model.load_state_dict(payload["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in payload:
        optimizer.load_state_dict(payload["optimizer_state_dict"])
    if scheduler is not None and "scheduler_state_dict" in payload:
        scheduler.load_state_dict(payload["scheduler_state_dict"])
    return payload


## Decoding

In [ ]:
def make_src_mask(src: torch.Tensor) -> torch.Tensor:
    return torch.ones(src.size(0), 1, src.size(1), dtype=torch.bool, device=src.device)


@torch.no_grad()
def greedy_decode(model: EncoderDecoder, src: torch.Tensor, max_len: int = TGT_LEN, bos_id: int = BOS_ID) -> torch.Tensor:
    model.eval()
    src_mask = make_src_mask(src)
    memory = model.encode(src, src_mask)
    ys = torch.full((src.size(0), 1), bos_id, dtype=torch.long, device=src.device)

    for _ in range(max_len):
        tgt_mask = subsequent_mask(ys.size(1), device=src.device)
        out = model.decode(memory, src_mask, ys, tgt_mask)
        logits = model.generator(out[:, -1])
        next_token = logits.argmax(dim=-1, keepdim=True)
        ys = torch.cat([ys, next_token], dim=1)

    return ys[:, 1:]


def sample_next_token(logits: torch.Tensor, temperature: float = 1.0, top_k: Optional[int] = 50) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    logits = logits / temperature
    if top_k is not None and top_k > 0:
        values, indices = torch.topk(logits, k=min(top_k, logits.size(-1)), dim=-1)
        probs = torch.softmax(values, dim=-1)
        sampled = torch.multinomial(probs, num_samples=1)
        return indices.gather(-1, sampled)

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


@torch.no_grad()
def sample_decode(model: EncoderDecoder, src: torch.Tensor, max_len: int = TGT_LEN, bos_id: int = BOS_ID, temperature: float = 1.0, top_k: Optional[int] = 50) -> torch.Tensor:
    model.eval()
    src_mask = make_src_mask(src)
    memory = model.encode(src, src_mask)
    ys = torch.full((src.size(0), 1), bos_id, dtype=torch.long, device=src.device)

    for _ in range(max_len):
        tgt_mask = subsequent_mask(ys.size(1), device=src.device)
        out = model.decode(memory, src_mask, ys, tgt_mask)
        logits = model.generator(out[:, -1])
        next_token = sample_next_token(logits, temperature=temperature, top_k=top_k)
        ys = torch.cat([ys, next_token], dim=1)

    return ys[:, 1:]


@torch.no_grad()
def generate_story_continuation(
    model: EncoderDecoder,
    tokenizer: Tokenizer,
    prompt: str,
    new_chunks: int = 2,
    strategy: str = "greedy",
    temperature: float = 1.0,
    top_k: int = 50,
    device: torch.device = DEVICE,
) -> str:
    model.eval()
    prompt_ids = tokenizer.encode(prompt).ids
    if len(prompt_ids) == 0:
        prompt_ids = [BOS_ID]

    story = list(prompt_ids)

    for _ in range(new_chunks):
        src_ids = story[-SRC_LEN:]
        src = torch.tensor([src_ids], dtype=torch.long, device=device)
        if strategy == "greedy":
            out_ids = greedy_decode(model, src, max_len=TGT_LEN).squeeze(0).tolist()
        else:
            out_ids = sample_decode(model, src, max_len=TGT_LEN, temperature=temperature, top_k=top_k).squeeze(0).tolist()
        story.extend(out_ids)

    return tokenizer.decode(story, skip_special_tokens=True)


## Training loop

In [ ]:
def build_ready_model_and_tokenizer():
    tokenizer = train_or_load_tokenizer()
    model = make_model(
        CONFIG.src_vocab,
        CONFIG.tgt_vocab,
        N=CONFIG.N,
        d_model=CONFIG.d_model,
        d_ff=CONFIG.d_ff,
        h=CONFIG.h,
        dropout=CONFIG.dropout,
    ).to(DEVICE)
    return model, tokenizer


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def print_model_summary(model: nn.Module):
    total = count_parameters(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total:,}")
    print(f"Trainable params: {trainable:,}")
    print(f"Device: {DEVICE}")
    if DEVICE.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"AMP (mixed precision): {'on' if AMP_ENABLED else 'off'}")


def fit(
    model: EncoderDecoder,
    tokenizer: Tokenizer,
    train_loader,
    val_loader,
    epochs: int = 10,
    warmup_steps: int = 4000,
    lr: float = 3e-4,
    weight_decay: float = 0.01,
    accum_iter: int = 1,
    grad_clip: float = 1.0,
    ckpt_name: str = "tinystories_transformer.pt",
    amp_enabled: bool = AMP_ENABLED,
):
    criterion = nn.CrossEntropyLoss(reduction="sum")
    optimizer, scheduler = build_optimizer_and_scheduler(
        model,
        d_model=CONFIG.d_model,
        warmup_steps=warmup_steps,
        lr=lr,
        weight_decay=weight_decay,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    history = []
    best_val = float("inf")
    best_path = CHECKPOINT_DIR / f"best_{ckpt_name}"
    last_path = CHECKPOINT_DIR / f"last_{ckpt_name}"
    train_state = TrainState()

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        train_stats = run_epoch(
            train_loader,
            model,
            criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            mode="train",
            accum_iter=accum_iter,
            grad_clip=grad_clip,
            train_state=train_state,
            device=DEVICE,
            scaler=scaler,
            amp_enabled=amp_enabled,
        )
        val_stats = run_epoch(
            val_loader,
            model,
            criterion,
            optimizer=None,
            scheduler=None,
            mode="eval",
            accum_iter=1,
            grad_clip=None,
            train_state=train_state,
            device=DEVICE,
            scaler=None,
            amp_enabled=amp_enabled,
        )

        row = {
            "epoch": epoch,
            "train_loss": train_stats["loss"],
            "train_ppl": train_stats["ppl"],
            "val_loss": val_stats["loss"],
            "val_ppl": val_stats["ppl"],
            "tokens_seen": train_stats["tokens"],
            "step": train_state.step,
        }
        history.append(row)

        print(
            f"train loss {row['train_loss']:.4f} | train ppl {row['train_ppl']:.2f} | "
            f"val loss {row['val_loss']:.4f} | val ppl {row['val_ppl']:.2f}"
        )

        save_checkpoint(
            last_path,
            model,
            TOKENIZER_PATH,
            optimizer=optimizer,
            scheduler=scheduler,
            config=asdict(CONFIG),
            history=history,
        )
        if row["val_loss"] < best_val:
            best_val = row["val_loss"]
            save_checkpoint(
                best_path,
                model,
                TOKENIZER_PATH,
                optimizer=optimizer,
                scheduler=scheduler,
                config=asdict(CONFIG),
                history=history,
            )
            print(f"saved new best checkpoint -> {best_path}")

    return history, best_path, last_path


## Train

In [ ]:
model, tokenizer = build_ready_model_and_tokenizer()
print_model_summary(model)

train_loader, val_loader = build_dataloaders(tokenizer)
history, best_path, last_path = fit(model, tokenizer, train_loader, val_loader, epochs=5)

## Generate

In [ ]:
prompt = "Once upon a time, there was"
print(generate_story_continuation(
    model,
    tokenizer,
    prompt,
    strategy="sample",
    new_chunks=4,
    temperature=0.9,
    top_k=50,
))